# 面试问题：LLM 持续/领域继续预训练怎样重热学习率、混合 Replay、避免灾难性遗忘，并设计 General/Domain 评测矩阵与发布门禁？

**一句话回答。** 持续预训练（Continual Pre-Training，CPT）不是“拿新领域语料再跑一次训练”：先从可追溯 checkpoint 恢复模型、优化器兼容信息与基线指标；切换分布后用短 warmup 把已衰减的学习率重新抬升到经过小模型实验确认的峰值；训练流中按 token 而不是按文件条数混入高质量旧域 replay；最后同时检查领域增益、通用能力遗忘、稳定性和数据合规，任何关键回归都阻止发布。

本 Notebook 用基础 PyTorch 手写一个微型因果 Bigram LM、稳定交叉熵、重新 warmup + cosine decay、可审计 replay 计划、灾难性遗忘度量和发布状态机。例子刻意让少量新域规则与旧域冲突，从而比较“只训领域数据”和“混入 replay”的差异。它展示的是机制与工程接口，不代表真实大模型规模上的最优超参数。

**主要资料。** [Continual Pre-Training of Large Language Models: How to (re)warm your model?](https://arxiv.org/abs/2308.04014) 研究新数据分布下的学习率重新 warmup；[Towards Effective and Efficient Continual Pre-training of Large Language Models](https://arxiv.org/abs/2407.18743) 强调数据混合、课程策略及通用与新能力联合评测。

In [ ]:
import copy  # 导入深拷贝工具，让不同训练策略从互不共享的基线权重开始。
import math  # 导入余弦函数，用于手写重新 warmup 后的学习率衰减。
import torch  # 导入 PyTorch 张量、自动微分和优化器基础能力。
from torch import nn  # 导入神经网络模块基类以手写微型因果语言模型。
torch.manual_seed(240)  # 固定随机种子，保证模型初始化和实验结果可以重复验证。
device = torch.device("cpu")  # 固定使用 CPU，确保普通环境也能冷启动执行。
vocabulary_size = 8  # 定义玩具词表规模，八个 token 足以表示旧域、新域与冲突。
assert torch.__version__  # 验证当前环境已经成功加载 PyTorch。
assert torch.initial_seed() == 240  # 验证实验使用了约定的随机种子。
assert device.type == "cpu"  # 验证示例没有偷偷依赖 GPU 环境。
assert vocabulary_size >= 8  # 验证词表能够容纳本实验设计的全部 token。

## 1. 先定义成功条件，再启动 CPT

面试回答应先明确基线：领域继续预训练希望降低新领域验证损失、提高领域任务准确率，但不能以不可接受的通用能力回归为代价。训练前应冻结一份基线 checkpoint，并记录 tokenizer 版本、数据快照、随机种子、优化器配置和 general/domain/safety 三类基线。没有基线就无法区分“领域变强”与“整体漂移”。

本例把 `general` 看作旧知识，把 `domain` 看作新领域。二者部分输入发生标签冲突，用来模拟术语含义或风格分布改变；新领域又包含旧数据从未见过的上下文，用来模拟真正的能力扩展。

In [ ]:
def rewarm_cosine_lr(step, total_steps, warmup_steps, peak_lr, floor_ratio):  # 定义线性重热与余弦衰减组合的学习率函数。
    assert 0 <= step < total_steps  # 验证当前训练步处在合法范围内。
    assert 0 < warmup_steps < total_steps  # 验证 warmup 长度既非零也不覆盖全部训练。
    assert peak_lr > 0.0  # 验证峰值学习率为正数。
    assert 0.0 < floor_ratio <= 1.0  # 验证末尾学习率比例位于合理区间。
    if step < warmup_steps:  # 判断当前是否仍处于重新升温阶段。
        return peak_lr * float(step + 1) / float(warmup_steps)  # 从较小值线性抬升到峰值学习率。
    decay_steps = total_steps - warmup_steps  # 计算余弦衰减阶段包含的更新次数。
    progress = float(step - warmup_steps) / float(max(decay_steps - 1, 1))  # 把衰减位置归一化到零到一。
    cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))  # 计算从一平滑下降到零的余弦系数。
    return peak_lr * (floor_ratio + (1.0 - floor_ratio) * cosine_factor)  # 在峰值与最小学习率之间进行插值。
probe_rates = [rewarm_cosine_lr(step, 20, 4, 0.08, 0.10) for step in range(20)]  # 生成一条完整探针曲线检查边界。
assert math.isclose(probe_rates[0], 0.02)  # 验证重热第一步从峰值的四分之一开始。
assert math.isclose(probe_rates[3], 0.08)  # 验证 warmup 末步到达指定峰值。
assert math.isclose(probe_rates[-1], 0.008)  # 验证最后一步下降到峰值的十分之一。
assert all(probe_rates[index] <= probe_rates[index + 1] for index in range(3))  # 验证重热阶段单调上升。
assert all(probe_rates[index] >= probe_rates[index + 1] for index in range(4, 19))  # 验证余弦阶段不会反向升高。

## 2. 为什么学习率需要重新 warmup

基础预训练结束时，学习率通常已经衰减到很低。若 CPT 原样继承这个末尾学习率，新领域损失下降会很慢；若第一步直接跳到很高峰值，又可能让参数产生剧烈更新。更稳妥的做法是从较低值线性重热，再按 cosine 衰减，并把峰值、warmup token 数和最终学习率视为需要消融的超参数。

论文观察到，重新升高学习率短期可能同时抬高旧域与新域损失，但较长训练后有利于新域收敛。因此线上监控不能看到前几百步 loss 上升就立即判断失败，也不能只看最终新域 loss；应保存阶段性 checkpoint，绘制两个分布的验证曲线，并限制更新范数。

In [ ]:
general_inputs = torch.tensor([0, 1, 2, 3, 4, 5] * 8, dtype=torch.long, device=device)  # 构造覆盖六个旧上下文的通用训练输入。
general_targets = torch.tensor([1, 2, 3, 4, 5, 0] * 8, dtype=torch.long, device=device)  # 为每个旧上下文指定稳定的下一 token。
domain_inputs = torch.tensor([6] * 8 + [7] * 8 + [0] + [1], dtype=torch.long, device=device)  # 构造以两个新上下文为主并含少量冲突的领域输入。
domain_targets = torch.tensor([7] * 8 + [6] * 8 + [6] + [7], dtype=torch.long, device=device)  # 指定新领域映射以及与旧域不同的两个目标。
data_sources = {"general": (general_inputs, general_targets), "domain": (domain_inputs, domain_targets)}  # 建立训练计划可按名称访问的数据源表。
def build_replay_plan(total_steps, replay_every):  # 构造确定性领域更新与旧域回放计划。
    assert total_steps > 0  # 验证训练计划至少包含一个更新步。
    assert replay_every >= 2  # 验证回放周期会同时保留领域更新。
    return ["general" if (step + 1) % replay_every == 0 else "domain" for step in range(total_steps)]  # 每到固定周期插入一次通用 replay。
replay_plan = build_replay_plan(total_steps=60, replay_every=4)  # 创建含百分之二十五旧域更新步的 CPT 计划。
assert general_inputs.numel() == general_targets.numel() == 48  # 验证通用输入与标签数量完全对齐。
assert domain_inputs.numel() == domain_targets.numel() == 18  # 验证领域输入与标签数量完全对齐。
assert set(data_sources) == {"general", "domain"}  # 验证训练账本只引用已登记的数据源。
assert replay_plan.count("general") == 15  # 验证六十步中准确插入十五步通用回放。
assert replay_plan.count("domain") == 45  # 验证剩余四十五步用于学习新领域。

## 3. Replay 要按 token 预算与来源治理

“混 20% 旧数据”必须说清分母。真实语料长度差异很大，按文档条数抽样会让长文档支配 token；工程上通常先做质量过滤、去重和许可证/隐私治理，再按训练 token 预算配比。Replay 不必复制完整旧语料，可以选择覆盖旧能力、低污染且可追溯的代表性样本。

本例用确定性计划让每四步中有一步读取 general replay，即 25% 的更新步。确定性计划便于单元测试；大规模训练可使用带种子的加权采样器，但仍要把实际消费 token 数、来源和重复率写入训练账本。

In [ ]:
class TinyBigramLM(nn.Module):  # 定义一个以转移矩阵实现的最小因果语言模型。
    def __init__(self, vocabulary_size):  # 接收词表大小并初始化下一 token 参数。
        super().__init__()  # 初始化 PyTorch 模块基类以登记所有可训练参数。
        self.transition_logits = nn.Embedding(vocabulary_size, vocabulary_size)  # 为每个当前 token 学习一行下一 token logits。
    def forward(self, input_ids):  # 根据当前 token 编号返回对应的下一 token 分布。
        return self.transition_logits(input_ids)  # 查询转移矩阵且不读取任何未来位置。
probe_model = TinyBigramLM(vocabulary_size).to(device)  # 创建探针模型用于核验接口。
probe_batch = torch.tensor([[0, 1, 2], [3, 4, 5]], dtype=torch.long, device=device)  # 构造两条三位置输入序列。
probe_logits = probe_model(probe_batch)  # 执行一次前向传播得到每个位置的词表 logits。
assert probe_logits.shape == torch.Size([2, 3, vocabulary_size])  # 验证输出满足批次、序列、词表三维约定。
assert sum(parameter.numel() for parameter in probe_model.parameters()) == vocabulary_size ** 2  # 验证模型参数就是完整转移矩阵。
assert torch.isfinite(probe_logits).all()  # 验证随机初始化没有产生非数值。
assert torch.allclose(probe_model(torch.tensor([0], device=device))[0], probe_model(torch.tensor([0], device=device))[0])  # 验证相同上下文查询得到确定性结果。

## 4. 从零实现微型因果语言模型

为了清楚看到遗忘发生在哪里，本例使用 `nn.Embedding(vocab_size, vocab_size)` 保存“当前 token 到下一 token logits”的转移矩阵。输入位置只查询自己的行，所以不读取未来 token，符合最小因果 next-token 接口。真实 Transformer 会在层间共享大量参数，遗忘往往更广泛；这里的可解释转移行让冲突与 replay 的作用容易核验。

面试时不要把这个玩具模型冒充完整 LLM。它缺少注意力、位置编码和长上下文能力，但仍保留三个关键对象：可训练参数、下一 token logits、由 labels 计算并反向传播的语言建模损失。

In [ ]:
def manual_cross_entropy(logits, targets):  # 手写稳定的多分类交叉熵以展示语言建模损失。
    row_max = logits.max(dim=-1, keepdim=True).values  # 取每行最大值用于消除指数溢出。
    shifted_logits = logits - row_max  # 平移 logits 且保持 softmax 概率不变。
    log_normalizer = torch.log(torch.exp(shifted_logits).sum(dim=-1, keepdim=True))  # 计算平移后的对数归一化因子。
    log_probabilities = shifted_logits - log_normalizer  # 得到每个候选下一 token 的对数概率。
    target_log_probabilities = log_probabilities.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # 取出正确目标 token 的对数概率。
    return -target_log_probabilities.mean()  # 对所有 token 的负对数概率求均值。
@torch.no_grad()  # 关闭评测阶段的梯度记录以减少无关状态。
def evaluate_source(model, inputs, targets):  # 同时计算指定数据源的损失与 top-1 准确率。
    logits = model(inputs)  # 使用候选模型计算下一 token logits。
    loss = manual_cross_entropy(logits, targets)  # 计算能反映概率置信度的平均交叉熵。
    accuracy = (logits.argmax(dim=-1) == targets).float().mean()  # 计算只比较最高分 token 的准确率。
    return {"loss": float(loss.item()), "accuracy": float(accuracy.item())}  # 转成普通数值便于写入评测矩阵。
uniform_logits = torch.zeros(3, vocabulary_size, device=device)  # 构造均匀分布 logits 检查手写损失。
uniform_targets = torch.tensor([0, 1, 2], dtype=torch.long, device=device)  # 构造三个合法的目标编号。
uniform_loss = manual_cross_entropy(uniform_logits, uniform_targets)  # 计算均匀词表预测的理论损失。
assert torch.allclose(uniform_loss, torch.log(torch.tensor(float(vocabulary_size))))  # 验证均匀分布损失等于词表大小的自然对数。
assert torch.isfinite(uniform_loss)  # 验证手写稳定形式得到有限数值。
assert evaluate_source(probe_model, general_inputs, general_targets)["loss"] > 0.0  # 验证随机模型的通用损失为正。
assert 0.0 <= evaluate_source(probe_model, domain_inputs, domain_targets)["accuracy"] <= 1.0  # 验证评测准确率处于合法范围。

## 5. 手写稳定交叉熵与评测函数

交叉熵先对每行 logits 减去最大值，再计算 log-sum-exp，避免大正数指数溢出；然后只取正确目标 token 的负对数概率并求均值。训练损失与发布指标要分开：loss 能感知置信度变化，accuracy 只看第一名。模型可能准确率暂时不变，但旧答案概率已经明显下降，所以遗忘门禁至少同时看 loss delta 与 accuracy drop。

真实 CPT 还要按语言、任务、长度、时间和安全类别分桶，报告置信区间，并固定评测模板与解码参数。这里先构造 general/domain 两列最小矩阵。

In [ ]:
def train_with_plan(model, plan, peak_lr, warmup_steps, floor_ratio=0.10):  # 定义不依赖 Trainer 的完整 CPT 更新循环。
    optimizer = torch.optim.SGD(model.parameters(), lr=peak_lr)  # 使用基础随机梯度下降器更新转移矩阵。
    history = []  # 初始化可审计训练历史以保存每步关键信号。
    for step, source_name in enumerate(plan):  # 按确定性数据计划逐步读取通用域或新领域。
        inputs, targets = data_sources[source_name]  # 根据来源名称取得当前完整小批次。
        current_lr = rewarm_cosine_lr(step, len(plan), warmup_steps, peak_lr, floor_ratio)  # 计算当前步骤重热或衰减后的学习率。
        for parameter_group in optimizer.param_groups:  # 遍历优化器参数组以同步当前学习率。
            parameter_group["lr"] = current_lr  # 把调度器结果显式写入优化器。
        optimizer.zero_grad(set_to_none=True)  # 清除上一步梯度并用空值降低无效写入。
        loss = manual_cross_entropy(model(inputs), targets)  # 对当前来源执行因果下一 token 训练损失。
        loss.backward()  # 从手写交叉熵反向传播到转移矩阵。
        squared_norm = sum(float((parameter.grad.detach() ** 2).sum().item()) for parameter in model.parameters())  # 汇总所有参数梯度平方和。
        gradient_norm = math.sqrt(squared_norm)  # 计算更新前的全局二范数用于监控。
        clip_scale = min(1.0, 1.0 / (gradient_norm + 1e-12))  # 计算最大范数为一的手工裁剪比例。
        for parameter in model.parameters():  # 遍历每个参数以执行显式梯度裁剪。
            parameter.grad.mul_(clip_scale)  # 原地缩放梯度并避免过大的单步更新。
        optimizer.step()  # 使用当前学习率和裁剪后梯度更新模型。
        history.append({"step": step, "source": source_name, "loss": float(loss.item()), "lr": current_lr, "grad_norm": gradient_norm})  # 记录可复查的训练账本。
    return history  # 返回完整历史供学习率、来源和稳定性检查。
base_model = TinyBigramLM(vocabulary_size).to(device)  # 初始化尚未学习任何数据的基础模型。
base_plan = ["general"] * 100  # 构造只使用通用数据的基础预训练计划。
base_history = train_with_plan(base_model, base_plan, peak_lr=0.60, warmup_steps=8)  # 训练得到后续 CPT 共同使用的通用 checkpoint。
baseline_general = evaluate_source(base_model, general_inputs, general_targets)  # 冻结 CPT 之前的通用能力基线。
baseline_domain = evaluate_source(base_model, domain_inputs, domain_targets)  # 记录尚未领域适配时的新域能力基线。
assert baseline_general["accuracy"] == 1.0  # 验证基础 checkpoint 已学会全部通用映射。
assert baseline_general["loss"] < 0.30  # 验证基础 checkpoint 对通用目标具有较高置信度。
assert baseline_domain["accuracy"] < 0.90  # 验证基础 checkpoint 尚未掌握完整新领域映射。
assert len(base_history) == 100  # 验证训练账本没有遗漏任何基础预训练步骤。
assert base_history[-1]["lr"] < base_history[7]["lr"]  # 验证基础预训练末尾学习率已经完成衰减。

## 6. 先训练通用基座并冻结基线

训练函数不依赖 Trainer：每一步选择一个数据源，手动清梯度、前向、反向、计算全局梯度范数、裁剪、设置当前学习率并更新参数。历史记录保存 step、source、loss、lr 和 grad norm，足以回答“重热真的发生了吗”“replay 实际插入了吗”。

基座只在 general 数据上预训练。得到稳定基线后必须深拷贝 checkpoint；若多个实验共享同一个可变模型对象，后续比较会被串扰，评测矩阵也就失去意义。

In [ ]:
domain_only_model = copy.deepcopy(base_model)  # 从冻结基线复制只训领域数据的对照模型。
replay_model = copy.deepcopy(base_model)  # 从同一基线复制带旧域回放的候选模型。
domain_only_plan = ["domain"] * 60  # 构造完全不含旧数据的持续预训练计划。
domain_only_history = train_with_plan(domain_only_model, domain_only_plan, peak_lr=0.45, warmup_steps=6)  # 用重热学习率执行 domain-only CPT。
replay_history = train_with_plan(replay_model, replay_plan, peak_lr=0.45, warmup_steps=6)  # 用相同曲线执行带百分之二十五 replay 的 CPT。
domain_only_general = evaluate_source(domain_only_model, general_inputs, general_targets)  # 测量只训领域数据后的旧能力。
domain_only_domain = evaluate_source(domain_only_model, domain_inputs, domain_targets)  # 测量只训领域数据后的新能力。
replay_general = evaluate_source(replay_model, general_inputs, general_targets)  # 测量带回放模型保留的旧能力。
replay_domain = evaluate_source(replay_model, domain_inputs, domain_targets)  # 测量带回放模型获得的新能力。
assert domain_only_history[0]["lr"] > base_history[-1]["lr"]  # 验证 CPT 首步已从基础训练末尾低学习率重新升温。
assert domain_only_history[5]["lr"] == 0.45  # 验证第六步准确达到 CPT 峰值学习率。
assert replay_history[0]["source"] == "domain"  # 验证回放计划先从新领域学习开始。
assert replay_history[3]["source"] == "general"  # 验证第四步按计划插入旧域 replay。
assert replay_domain["loss"] < baseline_domain["loss"]  # 验证带回放 CPT 仍然显著学习了新领域。
assert replay_general["loss"] < domain_only_general["loss"]  # 验证 replay 比纯领域训练更好地保留旧域概率。

## 7. 对照实验：Domain-only 与 Domain + Replay

两条 CPT 路径从完全相同的基线权重出发，并使用相同的总步数与重热曲线。唯一主要变量是数据计划：`domain_only` 每步只看新域；`with_replay` 每四步回放一次旧域。这样才能把保留能力的差异合理归因给 replay，而不是训练预算或初始化不同。

新域的大多数样本来自两个新上下文，少数样本与旧映射冲突。因此 replay 能在保留大部分旧映射的同时学会新上下文；遇到真正不可同时满足的冲突时，数据配比决定折中点，工程上还可以用路由、领域标签或参数高效模块隔离冲突。

In [ ]:
models = {"base_checkpoint": base_model, "domain_only": domain_only_model, "with_replay": replay_model}  # 登记三个候选以构造评测矩阵。
evaluation_sets = {"general": (general_inputs, general_targets), "domain": (domain_inputs, domain_targets)}  # 登记通用域与领域域两列固定评测集。
eval_matrix = {model_name: {set_name: evaluate_source(model, *dataset) for set_name, dataset in evaluation_sets.items()} for model_name, model in models.items()}  # 计算每个 checkpoint 在每个分布上的指标。
for model_name in eval_matrix:  # 遍历每个候选以补充相对基线的派生指标。
    eval_matrix[model_name]["general_loss_delta"] = eval_matrix[model_name]["general"]["loss"] - baseline_general["loss"]  # 计算通用损失相对基线的遗忘量。
    eval_matrix[model_name]["general_accuracy_drop"] = baseline_general["accuracy"] - eval_matrix[model_name]["general"]["accuracy"]  # 计算通用准确率下降幅度。
    eval_matrix[model_name]["domain_loss_gain"] = baseline_domain["loss"] - eval_matrix[model_name]["domain"]["loss"]  # 计算领域损失相对基线的改善量。
assert math.isclose(eval_matrix["base_checkpoint"]["general_loss_delta"], 0.0, abs_tol=1e-9)  # 验证基线相对自身没有遗忘。
assert eval_matrix["domain_only"]["general_loss_delta"] > eval_matrix["with_replay"]["general_loss_delta"]  # 验证纯领域训练造成更大旧域损失上升。
assert eval_matrix["with_replay"]["domain_loss_gain"] > 0.0  # 验证回放候选的领域能力相对基线确有收益。
assert eval_matrix["with_replay"]["general"]["accuracy"] >= eval_matrix["domain_only"]["general"]["accuracy"]  # 验证回放候选至少保留同等旧域 top-1 能力。
assert set(eval_matrix["with_replay"]["general"]) == {"loss", "accuracy"}  # 验证矩阵单元同时保留置信度与准确率指标。
assert all(math.isfinite(row["domain_loss_gain"]) for row in eval_matrix.values())  # 验证所有候选的领域增益均为有限数值。

## 8. General/Domain Eval Matrix 与遗忘量

评测矩阵的行是候选 checkpoint，列至少包含 general 与 domain。领域增益定义为“基线领域 loss 减去候选领域 loss”；通用遗忘可以用“候选 general loss 减去基线 general loss”和 accuracy drop 同时描述。正的领域增益越大越好，正的通用 loss delta 越大越危险。

不要把两列简单平均成一个总分，否则大幅通用回归可能被领域增益抵消。生产环境应为各关键切片设置独立硬门槛，再用综合分数做已通过候选之间的排序。还应与同 token 预算的从头训练、低学习率续训及不同 replay 比例做消融。

In [ ]:
def release_gate(candidate_row, baseline_domain_row, max_general_loss_delta, min_general_accuracy, min_domain_accuracy):  # 定义面向 CPT 候选的可解释发布门禁。
    reasons = []  # 初始化机器可读的阻断原因列表。
    if candidate_row["domain"]["loss"] >= baseline_domain_row["loss"]:  # 检查候选是否真的降低新领域损失。
        reasons.append("领域损失没有优于基线")  # 记录缺少领域收益的阻断原因。
    if candidate_row["domain"]["accuracy"] < min_domain_accuracy:  # 检查领域准确率是否达到最低业务要求。
        reasons.append("领域准确率未达门槛")  # 记录领域效果不足的阻断原因。
    if candidate_row["general_loss_delta"] > max_general_loss_delta:  # 检查通用概率退化是否超过允许预算。
        reasons.append("通用损失回归超限")  # 记录灾难性遗忘的阻断原因。
    if candidate_row["general"]["accuracy"] < min_general_accuracy:  # 检查通用 top-1 能力是否跌破硬底线。
        reasons.append("通用准确率跌破底线")  # 记录通用准确率回归的阻断原因。
    return {"status": "approved" if not reasons else "blocked", "reasons": reasons}  # 返回明确状态和全部失败条件。
loss_budget = 0.03  # 设置相对基线最多允许零点零三的通用损失增幅。
minimum_general_accuracy = 0.80  # 设置至少保留百分之八十通用准确率的硬门槛。
minimum_domain_accuracy = 0.85  # 设置领域准确率至少达到百分之八十五。
domain_only_gate = release_gate(eval_matrix["domain_only"], baseline_domain, loss_budget, minimum_general_accuracy, minimum_domain_accuracy)  # 对无 replay 候选执行发布判定。
replay_gate = release_gate(eval_matrix["with_replay"], baseline_domain, loss_budget, minimum_general_accuracy, minimum_domain_accuracy)  # 对带 replay 候选执行相同发布判定。
assert domain_only_gate["status"] == "blocked"  # 验证发生明显遗忘的纯领域候选被门禁阻止。
assert "通用损失回归超限" in domain_only_gate["reasons"]  # 验证阻断信息明确指出旧域概率回归。
assert replay_gate["status"] == "approved"  # 验证兼顾领域收益和旧能力的 replay 候选获准发布。
assert replay_gate["reasons"] == []  # 验证通过候选没有残留失败原因。
assert eval_matrix["with_replay"]["domain"]["accuracy"] >= minimum_domain_accuracy  # 验证获批候选满足领域准确率门槛。
assert eval_matrix["with_replay"]["general"]["accuracy"] >= minimum_general_accuracy  # 验证获批候选满足通用准确率门槛。
print({"domain_only": domain_only_gate, "with_replay": replay_gate})  # 输出精简门禁结果供学习者直观看到策略差异。

## 9. 发布门禁：训练完成不等于可以上线

门禁输入应是版本化评测产物，而不是人工抄写的单个数字。示例状态机检查：领域准确率是否达到最低值、领域 loss 是否确实优于基线、general loss 增幅是否超限、general accuracy 是否跌破底线。任一硬条件失败就返回 `blocked` 和机器可读原因；全部通过才返回 `approved`。

实际发布还要加入安全/隐私、偏见、事实性、延迟、吞吐、模型签名、训练数据 manifest 和回滚演练。若门禁失败，首选动作是回到学习率峰值、replay 比例、冲突数据和 checkpoint 区间做诊断，而不是临时降低阈值。